# 09 - Auditoría de la Ejecución

Trabajo Práctico - Pipeline ETL end-to-end (Bronze → Silver / DQL → Gold)

Última tarea del pipeline. Revisa el estado final de **todas** las tablas y lo registra en `datos_empresas.auditoria.de_pipeline_ejecuciones`, en la fila `(id_ejecucion, tabla)` donde `escribir_merge` ya anotó las filas insertadas, actualizadas y eliminadas.

## Validaciones (si alguna falla, la tarea falla y el Job queda en rojo)

| Validación | Qué demuestra |
|---|---|
| `filas_duplicadas = 0` en Silver, DQL y Gold | La llave de negocio es única: el pipeline no duplica registros. |
| Bronze = Silver + DQL en cada fuente | Ningún registro de Bronze se pierde ni se cuenta dos veces: todos terminan aceptados o rechazados. |
| Filas de la dimensión = filas de Silver RUES y filas del hecho = filas de Silver SECOP | Gold proviene íntegramente de Silver. |

El `checksum` (suma de `xxhash64(_hash_registro)`) es una huella del contenido de cada tabla. Si dos ejecuciones sobre el mismo Bronze dan el mismo checksum, las tablas son idénticas.

In [ ]:
%run ./00_Utilidades_Pipeline

---
## Paso 1: Métricas de estado de cada tabla

In [ ]:
%python
TABLAS_A_AUDITAR = []  # (capa, tabla, llaves, columna_hash)
for f in FUENTES.values():
    TABLAS_A_AUDITAR.append(("bronze", f["bronze"], None, None))
    TABLAS_A_AUDITAR.append(("silver", f["silver"], f["llaves"], "_hash_registro"))
    TABLAS_A_AUDITAR.append(("dql", f["dql"], ["_id_registro_bronze"], "_hash_registro"))
for t, llaves in TABLAS_GOLD.items():
    TABLAS_A_AUDITAR.append(("gold", t, llaves, "_hash_registro"))

metricas = {}
for capa, t, llaves, col_hash in TABLAS_A_AUDITAR:
    df = spark.table(t)
    exprs = [F.count("*").alias("filas_tabla")]
    if llaves:
        exprs.append(F.countDistinct(*[F.col(f"`{c}`") for c in llaves]).alias("llaves_distintas"))
        exprs.append(F.sum(F.xxhash64(col_hash).cast("decimal(38,0)")).alias("checksum"))
    fila = df.agg(*exprs).collect()[0].asDict()
    fila.setdefault("llaves_distintas", fila["filas_tabla"])
    fila.setdefault("checksum", None)
    fila["filas_duplicadas"] = fila["filas_tabla"] - fila["llaves_distintas"]
    fila["capa"] = capa
    metricas[t] = fila

---
## Paso 2: Validaciones

In [ ]:
%python
errores = []

for t, m in metricas.items():
    if m["capa"] != "bronze" and m["filas_duplicadas"] != 0:
        errores.append(f"{t}: {m['filas_duplicadas']} filas con llave duplicada")

for nombre, f in FUENTES.items():
    b, s, d = (metricas[f[k]]["filas_tabla"] for k in ("bronze", "silver", "dql"))
    if b != s + d:
        errores.append(f"[{nombre}] Bronze ({b}) != Silver ({s}) + DQL ({d})")

if metricas[GOLD_DIM_EMPRESAS]["filas_tabla"] != metricas[SILVER_RUES]["filas_tabla"]:
    errores.append("gold.de_dim_empresas no tiene una fila por empresa de Silver")
if metricas[GOLD_FACT_CONTRATOS]["filas_tabla"] != metricas[SILVER_SECOP]["filas_tabla"]:
    errores.append("gold.de_fact_contratos no tiene una fila por contrato de Silver")

validacion_ok = not errores
print("VALIDACIÓN OK" if validacion_ok else "VALIDACIÓN CON ERRORES:\n - " + "\n - ".join(errores))

---
## Paso 3: Registrar en la tabla de auditoría (MERGE por `id_ejecucion` + `tabla`)

In [ ]:
%python
ahora = datetime.now()
for t, m in metricas.items():
    registrar_auditoria(t, m["capa"], {
        "filas_tabla": int(m["filas_tabla"]),
        "llaves_distintas": int(m["llaves_distintas"]),
        "filas_duplicadas": int(m["filas_duplicadas"]),
        "checksum": m["checksum"],
        "validacion_ok": validacion_ok,
        "fecha_validacion": ahora,
    })

display(spark.sql(f"""
    SELECT capa, tabla, filas_tabla, filas_duplicadas, filas_insertadas, filas_actualizadas, filas_eliminadas,
           checksum, validacion_ok
    FROM {AUDITORIA_EJECUCIONES}
    WHERE id_ejecucion = '{ID_EJECUCION}'
    ORDER BY CASE capa WHEN 'bronze' THEN 1 WHEN 'silver' THEN 2 WHEN 'dql' THEN 3 ELSE 4 END, tabla
"""))

In [ ]:
%python
if not validacion_ok:
    raise AssertionError("Validaciones del pipeline fallidas: " + "; ".join(errores))

finalizar({"notebook": "09_Auditoria_Ejecucion", "id_ejecucion": ID_EJECUCION, "validacion_ok": validacion_ok})